In [10]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


# MINE at Retrieval! 
MINE is a tool to evaluate the quality of an LLM for knowledge graph creation. 
It generates a knowledge graph and then tries to retrieve data from it. 

MINER, MINE at Retrieval is like MINE but operating purely at the retrieval stage. 
Knowledge ingestion and retrieval are performed separately to the tool, allowing it to evaluate multiple knowledge storage techniques. 
Put simply, it's one level up in abstraction. 

This can be run for anything that takes some text as an input query and outputs some text containing information related to it. 
We may want to change the dataset a bit, as the queries are not in the form of questions. 


In [11]:
# Implement this for a system
class MINER(object):
	async def ingest(self, text: str):
		""" Ingest knowledge from some text. """
		pass
	async def pre_retrieve(self):
		pass
	async def retrieve(self, text: str) -> str:
		""" Find information relevant to a text. """
		pass
	async def reset(self):
		""" Forget ingested knowledge. """
		pass


In [12]:
from pathlib import Path

# Dataset in use 
# MINE_DIRECTORY = Path("../datasets/MINE")
MINE_DIRECTORY = Path("../datasets/paper")
# Storage location for results
# RESULTS_DIR = Path("/tmp/miner")
RESULTS_DIR = Path("./results_paper")


In [13]:
import json
import time
import dspy


class EvalSignature(dspy.Signature):
	""" 
	ROLE: You are an evaluator that checks if the statement can be deduced from the information in the context.
	TASK: Determine whether the context contains the information stated in the statement.
	"""
	context: str = dspy.InputField()
	statement: str = dspy.InputField()
	context_contains_statement: bool = dspy.OutputField()
eval = dspy.Predict(EvalSignature)


# Concurrency is not used here for now
# Maaaybe later 
# If I am annoyed enough! 
# TODO: timing is influenced by caching and I'm not sure what to do about it
async def miner_evaluate_individual(miner: MINER, judge_model: str, limit: int | None = 8):
	result = []
	paths = list(MINE_DIRECTORY.iterdir())
	if limit:
		print(f"WARNING: limiting to {limit} samples")
		paths = paths[:limit]
	for i, p in enumerate(paths):
		print(f"Evaluate {p.name} ({i+1}/{len(paths)})")
		# Load data 
		with open(p, "r") as fp:
			mine_data = json.load(fp)
		
		# Ingest text 
		print("Ingest...")
		ingest_st = time.time()
		await miner.ingest(mine_data["essay"])
		await miner.pre_retrieve()
		ingest_en = time.time()

		# Query and evaluate 
		queries = []
		with dspy.context(lm=dspy.LM(judge_model)):
			for i, a in enumerate(mine_data["answers"]):
				print(f"\rQuery {i+1}/{len(mine_data["answers"])}", end="")
				q_st = time.time()
				info = await miner.retrieve(a)
				q_en = time.time()
				contained = (await eval.acall(context=info, statement=a)).context_contains_statement
				queries.append({
					"query": a,
					"context": info,
					"contained": contained,
					"duration": q_en - q_st,
				})
			print()
		result.append({
			"filename": p.name,
			"ingest_duration": ingest_en - ingest_st,
			"queries": queries,
		})
		await miner.reset()
	return result


async def miner_evaluate_aggregate(miner: MINER, judge_model: str, limit: int | None = 8):
	result = []
	paths = list(MINE_DIRECTORY.iterdir())
	if limit:
		print(f"WARNING: limiting to {limit} samples")
		paths = paths[:limit]
	for i, p in enumerate(paths):
		print(f"Ingest {p.name} ({i+1}/{len(paths)})")
		# Load data 
		with open(p, "r") as fp:
			mine_data = json.load(fp)
		
		# Ingest text 
		print("Ingest...")
		ingest_st = time.time()
		await miner.ingest(mine_data["essay"])
		await miner.pre_retrieve()
		ingest_en = time.time()

	for i, p in enumerate(paths):
		print(f"Evaluate {p.name} ({i+1}/{len(paths)})")
		with open(p, "r") as fp:
			mine_data = json.load(fp)

		queries = []
		with dspy.context(lm=dspy.LM(judge_model)):
			for i, a in enumerate(mine_data["answers"]):
				print(f"\rQuery {i+1}/{len(mine_data["answers"])}", end="")
				q_st = time.time()
				info = await miner.retrieve(a)
				q_en = time.time()
				contained = (await eval.acall(context=info, statement=a)).context_contains_statement
				queries.append({
					"query": a,
					"context": info,
					"contained": contained,
					"duration": q_en - q_st,
				})
			print()
		result.append({
			"filename": p.name,
			"queries": queries,
		})
	await miner.reset()
	return result


# Assumes that the dataset has an array with chunk texts called "chunks"
async def miner_evaluate_chunked(miner: MINER, judge_model: str):
	result = []
	paths = list(MINE_DIRECTORY.iterdir())
	for i, p in enumerate(paths):
		print(f"Evaluate {p.name} ({i+1}/{len(paths)})")
		# Load data 
		with open(p, "r") as fp:
			mine_data = json.load(fp)
		
		# Ingest text 
		print("Ingest...")
		ingest_st = time.time()
		for i, text in enumerate(mine_data["chunks"]):
			print(f"Chunk {i+1}/{len(mine_data["chunks"])}")
			await miner.ingest(text)
		await miner.pre_retrieve()
		ingest_en = time.time()

		# Query and evaluate 
		queries = []
		with dspy.context(lm=dspy.LM(judge_model)):
			for i, a in enumerate(mine_data["answers"]):
				print(f"\rQuery {i+1}/{len(mine_data["answers"])}", end="")
				q_st = time.time()
				info = await miner.retrieve(a)
				q_en = time.time()
				contained = (await eval.acall(context=info, statement=a)).context_contains_statement
				queries.append({
					"query": a,
					"context": info,
					"contained": contained,
					"duration": q_en - q_st,
				})
			print()
		result.append({
			"filename": p.name,
			"ingest_duration": ingest_en - ingest_st,
			"queries": queries,
		})
		await miner.reset()
	return result


In [14]:
import asyncio
from typing import Any
from aiolimiter import AsyncLimiter
from prettytable import PrettyTable


async def evaluate(
	items: list[tuple[str, Any]],
	concurrency: int = 3,
):
	"""
	Runs some evaluation functions (potentially concurrently) and saves the resulting output data to a file. 
	"""

	limiter = AsyncLimiter(concurrency)
	async def limited(f):
		async with limiter:
			return await f
	name_to_path = lambda n: RESULTS_DIR / f"{n}.json"

	names, tasks = zip(*items)
	paths = [name_to_path(n) for n in names]

	print(f"Running {len(names)} evaluations with concurrency {concurrency}")
	if concurrency > 1:
		tasks = [limited(f) for f in tasks]
		results = await asyncio.gather(*tasks)
	else: 
		print("(this will be run in serial, just so you know)")
		results = [await f for f in tasks]
	print("Done!")

	RESULTS_DIR.mkdir(exist_ok=True)
	for name, path, result in zip(names, paths, results):
		print(f"Saving '{path}'")
		with open(path, "w") as fp:
			json.dump({
				"name": name,
				"result": result,
			}, fp, indent=2)
	return paths


def score_count(result):
	""" Computes total score and count. """
	score = 0
	count = 0
	for part in result:
		for query in part["queries"]:
			count += 1
			score += int(query["contained"])
	return score, count


def response_length(result):
	""" 
	For each "correct" response, how long is it? 
	
	Note: Having only one correct response that is concise will give a "good" score for this. 
	"""
	length = 0
	count = 0
	for part in result:
		for query in part["queries"]:
			if query["contained"]:
				length += len(query["context"])
				count += 1
	return length / count


def mean_median_query_time(result):
	times = []
	for part in result:
		for query in part["queries"]:
			times.append(query["duration"])
	mean = sum(times) / len(times)
	times.sort()
	median = times[len(times)//2]
	return mean, median


def show_results():
	table = PrettyTable()
	table.field_names = [
		"Name", 
		"Score", 
		"Context Length", 
		"Conciseness",
		# "Query Duration (mean)", 
		# "Query Duration (median)",
	]
	RESULTS_DIR.mkdir(exist_ok=True)
	for f in RESULTS_DIR.iterdir():
		with open(f, "r") as fp:
			data = json.load(fp)
		score, count = score_count(data["result"])
		rlen = response_length(data["result"])
		# mean, median = mean_median_query_time(data["result"])
		table.add_row([
			data["name"], 
			f"{score/count*100:.2f}% ({score}/{count})", 
			f"{rlen:.2f}",
			f"{score/count*100/rlen:.2f}",
			# f"{mean:.2f}s",
			# f"{median:.2f}s",
		])
	print(table)
		

# System Implementations
Something to test with, reference implementations. 

In [15]:
processing_model = "bedrock/us.anthropic.claude-opus-4-5-20251101-v1:0"
embedding_model = "bedrock/amazon.titan-embed-text-v2:0"
judge_model = "bedrock/us.amazon.nova-pro-v1:0"


In [16]:
class FullContextMINER(MINER):
	kb = ""

	def __init__(
		self,
	):
		pass

	async def ingest(self, text: str):
		self.kb += text + "\n"
	
	async def pre_retrieve(self):
		pass

	async def retrieve(self, text: str) -> str:
		return self.kb

	async def reset(self):
		self.kb = ""

# await evaluate(
# 	[
# 		("full_context", miner_evaluate_individual(FullContextMINER(), judge_model)),
# 	],
# 	1,
# )
show_results()

+------+-------+----------------+-------------+
| Name | Score | Context Length | Conciseness |
+------+-------+----------------+-------------+
+------+-------+----------------+-------------+


In [17]:
import litellm
import numpy as np


class BasicVectorMINER(MINER):
	""" A MINER implementation for a very simple vector RAG system. """

	chunks: list[str] = []
	embeddings: list = []

	def __init__(
		self,
		chunk_size: int = 200,
		overlap: int = 20,
		quantile: float = 0.95,
		embedding_model: str = "bedrock/amazon.titan-embed-text-v2:0",
	):
		self.chunk_size = chunk_size
		self.overlap = overlap
		self.quantile = quantile 
		self.embedding_model = embedding_model

	async def ingest(self, text: str):
		new_chunks = [text[i*self.chunk_size:(i+1)*self.chunk_size+self.overlap] for i in range(0, len(text)//self.chunk_size)]
		new_embeddings = await litellm.aembedding(input=new_chunks, model=self.embedding_model)
		new_embeddings = [np.array(e.embedding) for e in new_embeddings.data]

		self.chunks += new_chunks
		self.embeddings += new_embeddings
	
	async def pre_retrieve(self):
		pass

	async def retrieve(self, text: str) -> str:
		# Make embedding 
		text_embedding = (await litellm.aembedding(self.embedding_model, input=[text]))
		text_embedding = np.array(text_embedding.data[0].embedding)

		# Find similarities 
		cosine = np.abs(np.dot(self.embeddings, text_embedding) / (np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(text_embedding)))
		sims = np.argsort(cosine)[::-1]

		# Find relevant 
		cutoff = np.quantile(sims, self.quantile)
		where = sims >= cutoff
		indices = np.nonzero(where)[0]

		# No need for ordering 
		return "\n".join([self.chunks[i] for i in indices])

	async def forget(self):
		self.chunks = []
		self.embeddings = []

# miner = BasicVectorMINER()
# result = await miner_evaluate_individual(miner, "bedrock/us.amazon.nova-pro-v1:0")
# score_count(result)

# await evaluate(
# 	[
# 		("vector_200_20_95", miner_evaluate_individual(BasicVectorMINER(), judge_model)),
# 		("vector_200_20_85", miner_evaluate_individual(BasicVectorMINER(quantile=0.85, embedding_model=embedding_model), judge_model)),
# 		("vector_100_20_95", miner_evaluate_individual(BasicVectorMINER(chunk_size=100, embedding_model=embedding_model), judge_model)),
# 		("vector_100_20_85", miner_evaluate_individual(BasicVectorMINER(chunk_size=100, quantile=0.85, embedding_model=embedding_model), judge_model)),
# 	],
# 	3,
# )
show_results()

+------+-------+----------------+-------------+
| Name | Score | Context Length | Conciseness |
+------+-------+----------------+-------------+
+------+-------+----------------+-------------+


In [18]:

# Implement this for a system
from hypergraph import aggregate, extract_edges_entities, query


class HypergraphMINER(MINER):
	kb = []
	eb = []
	hg = None

	def __init__(
		self,
		# Settings from paper
		kv: int = 60,
		tv: int = 50,
		kh: int = 60,
		th: int = 5,
		# This model doesn't make an error! Many do... 
		model: str = "bedrock/us.anthropic.claude-opus-4-5-20251101-v1:0",
		embedding_model: str = "bedrock/amazon.titan-embed-text-v2:0",
	):
		self.kv = kv
		self.tv = tv
		self.kh = kh
		self.th = th
		self.model = model
		self.embedding_model = embedding_model
	
	async def ingest(self, text: str):
		with dspy.context(lm=dspy.LM(self.model)):
			k, e = await extract_edges_entities(text)	
		self.kb += k
		self.eb += e
	
	async def pre_retrieve(self):
		with dspy.context(lm=dspy.LM(self.model)):
			self.hg = await aggregate(self.kb, self.eb, self.embedding_model)

	async def retrieve(self, text: str) -> str:
		assert not (self.hg is None)
		with dspy.context(lm=dspy.LM(self.model)):
			k = await query(text, self.hg, self.embedding_model, kv=self.kv, tv=self.tv, kh=self.kh, th=self.th)
		return k

	async def reset(self):
		self.kb = []
		self.eb = []
		self.hg = None


await evaluate(
	[
		("hgr_60_50_60_5_c45o", miner_evaluate_individual(HypergraphMINER(60, 50, 60, 5, model=processing_model, embedding_model=embedding_model), processing_model)),
		("hgr_60_30_60_3_c45o", miner_evaluate_individual(HypergraphMINER(60, 30, 60, 3, model=processing_model, embedding_model=embedding_model), processing_model)),
		("hgr_60_50_60_5_aggregate", miner_evaluate_aggregate(HypergraphMINER(60, 50, 60, 5, model=processing_model, embedding_model=embedding_model), processing_model)),
		("hgr_60_30_60_3_aggregate", miner_evaluate_aggregate(HypergraphMINER(60, 30, 60, 3, model=processing_model, embedding_model=embedding_model), processing_model)),
	],
	1,
)
show_results()

Running 4 evaluations with concurrency 1
(this will be run in serial, just so you know)
Evaluate A Comprehensive Survey of Continual Learning  Theory, Method and Application.json (1/8)
Ingest...


Aggregate into graph
Generate embeddings
Add entities
Add edges
Query 1/15Found 25 entities
Found 15 hyperedges
Selected 0 entities
Selected 4 hyperedges
Have 4 information pieces
Fusion expand to 9 information pieces
Query 2/15Found 25 entities
Found 15 hyperedges
Selected 0 entities
Selected 0 hyperedges
Have 0 information pieces
Fusion expand to 0 information pieces
Query 3/15Found 25 entities
Found 15 hyperedges
Selected 1 entities
Selected 1 hyperedges
Have 2 information pieces
Fusion expand to 2 information pieces
Query 4/15Found 25 entities
Found 15 hyperedges
Selected 1 entities
Selected 1 hyperedges
Have 2 information pieces
Fusion expand to 2 information pieces
Query 5/15Found 25 entities
Found 15 hyperedges
Selected 0 entities
Selected 1 hyperedges
Have 1 information pieces
Fusion expand to 2 information pieces
Query 6/15Found 25 entities
Found 15 hyperedges
Selected 0 entities
Selected 0 hyperedges
Have 0 information pieces
Fusion expand to 0 information pieces
Query 7/15Fo

In [19]:
import kggen


class KGv2MINER(MINER):
	kb: list[tuple[str, str, str]] = []
	eb: list[str] = []
	kg = None

	def __init__(
		self,
		model: str = "bedrock/us.anthropic.claude-opus-4-5-20251101-v1:0",
		embedding_model: str = "bedrock/amazon.titan-embed-text-v2:0",
	):
		self.model = model
		self.embedding_model = embedding_model
	
	async def ingest(self, text: str):
		with dspy.context(lm=dspy.LM(self.model)):
			e, k = await kggen.extract(text)
		self.kb += k
		self.eb += e
	
	async def pre_retrieve(self):
		with dspy.context(lm=dspy.LM(self.model)):
			self.eb, self.kb = await kggen.resolve(self.eb, self.kb)
			self.kg = await kggen.make_graph(self.eb, self.kb, self.embedding_model)

	async def retrieve(self, text: str) -> str:
		assert not (self.kg is None)
		with dspy.context(lm=dspy.LM(self.model)):
			k = await kggen.query(text, self.kg, self.embedding_model)
		return "\n".join([" ".join(t) for t in k])

	async def reset(self):
		self.kb = []
		self.eb = []
		self.kg = None


# await evaluate(
# 	[
# 		("kg-gen", miner_evaluate_individual(KGv2MINER(model=processing_model, embedding_model=embedding_model), judge_model)),
# 	],
# 	1,
# )
show_results()


+--------------------------+-----------------+----------------+-------------+
|           Name           |      Score      | Context Length | Conciseness |
+--------------------------+-----------------+----------------+-------------+
|   hgr_60_50_60_5_c45o    | 29.17% (35/120) |     769.80     |     0.04    |
|   hgr_60_30_60_3_c45o    | 40.00% (48/120) |    1654.27     |     0.02    |
| hgr_60_50_60_5_aggregate | 29.17% (35/120) |    1233.06     |     0.02    |
| hgr_60_30_60_3_aggregate | 39.17% (47/120) |    4627.70     |     0.01    |
+--------------------------+-----------------+----------------+-------------+


In [20]:
show_results()

+--------------------------+-----------------+----------------+-------------+
|           Name           |      Score      | Context Length | Conciseness |
+--------------------------+-----------------+----------------+-------------+
|   hgr_60_50_60_5_c45o    | 29.17% (35/120) |     769.80     |     0.04    |
|   hgr_60_30_60_3_c45o    | 40.00% (48/120) |    1654.27     |     0.02    |
| hgr_60_50_60_5_aggregate | 29.17% (35/120) |    1233.06     |     0.02    |
| hgr_60_30_60_3_aggregate | 39.17% (47/120) |    4627.70     |     0.01    |
+--------------------------+-----------------+----------------+-------------+


In [21]:
# Re-score the hgr results because they're sus

# Query and evaluate 
async def rescore(path: Path, name: str, judge_model: str):
	with open(path, "r") as fp:
		data = json.load(fp)
	results = []
	for i, result in enumerate(data["result"]):
		print(f"Result {i+1}/{len(data["result"])}")
		queries = []
		for q in result["queries"]:
			query = q["query"]
			con = q["context"]
			with dspy.context(lm=dspy.LM(judge_model)):
				contained = (await eval.acall(context=con, statement=query)).context_contains_statement
				queries.append({
					"query": query,
					"context": con,
					"contained": contained,
					# "duration": q_en - q_st,
				})
		results.append({
			"filename": result["filename"],
			# "ingest_duration": ingest_en - ingest_st,
			"queries": queries,
		})
	data = {
		"name": name,
		"result": results,
	}

	RESULTS_DIR.mkdir(exist_ok=True)
	out_path = RESULTS_DIR / f"{name}.json"
	with open(path, "w") as fp:
		json.dump(data, fp, indent=2)
	
	return data, out_path

# _, _ = await rescore(Path("./results_paper/hgr_60_30_60_3.json"), "hgr_rescore", processing_model)


In [22]:
show_results()

+--------------------------+-----------------+----------------+-------------+
|           Name           |      Score      | Context Length | Conciseness |
+--------------------------+-----------------+----------------+-------------+
|   hgr_60_50_60_5_c45o    | 29.17% (35/120) |     769.80     |     0.04    |
|   hgr_60_30_60_3_c45o    | 40.00% (48/120) |    1654.27     |     0.02    |
| hgr_60_50_60_5_aggregate | 29.17% (35/120) |    1233.06     |     0.02    |
| hgr_60_30_60_3_aggregate | 39.17% (47/120) |    4627.70     |     0.01    |
+--------------------------+-----------------+----------------+-------------+
